In [1]:
# [Cell 0] 재현성 설정, 데이터 로드 및 중복 측정 정제
import random
import numpy as np
import pandas as pd
import tensorflow as tf

# 1. 재현성(Reproducibility) 확보를 위한 Global Seed 고정
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# 2. 데이터 로드
normal_df = pd.read_csv('press_data_normal.csv')
outlier_df = pd.read_csv('outlier_data.csv')

# 3. CSV 저장 시 생성된 불필요한 저장용 인덱스 칼럼 제거
normal_df = normal_df.loc[:, ~normal_df.columns.str.contains('^Unnamed')]
outlier_df = outlier_df.loc[:, ~outlier_df.columns.str.contains('^Unnamed')]

# 4. 저장용 인덱스를 제외한 실제 데이터 기준 중복 1건 제거
duplicate_cols = ['TimeStamp', 'AI0_Vibration', 'AI1_Vibration', 'AI2_Current', 'Equipment_state']

print(f" 정제 전 정상 데이터: {len(normal_df):,}행 | 이상 데이터: {len(outlier_df):,}행")

normal_df = normal_df.drop_duplicates(subset=duplicate_cols, keep='first').reset_index(drop=True)
outlier_df = outlier_df.drop_duplicates(subset=duplicate_cols, keep='first').reset_index(drop=True)

print(f" 정제 후 정상 데이터: {len(normal_df):,}행 | 이상 데이터: {len(outlier_df):,}행")
print(" Random Seed 고정 완료 (Python, NumPy, TensorFlow = 42)")

 정제 전 정상 데이터: 20,000행 | 이상 데이터: 600행
 정제 후 정상 데이터: 19,999행 | 이상 데이터: 600행
 Random Seed 고정 완료 (Python, NumPy, TensorFlow = 42)


In [2]:
# [Cell 1] 시계열 공백 분리, Rolling Feature 생성 및 시간순 Hold-out Split
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

sensor_cols = ['AI0_Vibration', 'AI1_Vibration', 'AI2_Current']
window_size = 5  # 기본 window_size

def generate_rolling_features(df, w_size=5, gap_threshold=0.5):
    """
    TimeStamp가 존재하는 경우 0.5초 초과 공백에서 Segment를 분리하고,
    TimeStamp가 없는 Mode 1 수동 입력은 하나의 연속 구간으로 처리한다.
    각 Segment 내부에서만 Rolling Feature를 생성한다.
    """
    df_out = df.copy()

    # TimeStamp가 있는 데이터 (Mode 2 및 학습 데이터)
    if 'TimeStamp' in df_out.columns:
        df_out['TimeStamp'] = pd.to_datetime(df_out['TimeStamp'])
        df_out = df_out.sort_values('TimeStamp').reset_index(drop=True)

        df_out['time_gap'] = df_out['TimeStamp'].diff().dt.total_seconds()
        df_out['segment'] = (
            df_out['time_gap'].fillna(0) > gap_threshold
        ).cumsum()

    # TimeStamp가 없는 Mode 1 수동 입력 데이터
    else:
        df_out = df_out.reset_index(drop=True)
        df_out['time_gap'] = 0.0
        df_out['segment'] = 0

    feature_cols = []

    for col in sensor_cols:
        grouped = df_out.groupby('segment')[col]
        roll_obj = grouped.rolling(w_size, min_periods=w_size)

        # Mean & Std
        df_out[f'{col}_roll_mean'] = roll_obj.mean().reset_index(
            level=0, drop=True
        )
        df_out[f'{col}_roll_std'] = roll_obj.std().reset_index(
            level=0, drop=True
        )

        # RMS
        sq_grouped = (df_out[col] ** 2).groupby(df_out['segment'])
        df_out[f'{col}_rms'] = np.sqrt(
            sq_grouped.rolling(w_size, min_periods=w_size)
            .mean()
            .reset_index(level=0, drop=True)
        )

        # PTP
        df_out[f'{col}_max'] = roll_obj.max().reset_index(level=0, drop=True)
        df_out[f'{col}_min'] = roll_obj.min().reset_index(level=0, drop=True)
        df_out[f'{col}_ptp'] = df_out[f'{col}_max'] - df_out[f'{col}_min']
        df_out.drop(columns=[f'{col}_max', f'{col}_min'], inplace=True)

        feature_cols.extend([
            col,
            f'{col}_roll_mean',
            f'{col}_roll_std',
            f'{col}_rms',
            f'{col}_ptp',
        ])

    feature_cols = sorted(list(set(feature_cols)))

    # Window Size 미충족으로 생성된 초기 NaN 행 제거
    df_out = df_out.dropna(subset=feature_cols).reset_index(drop=True)
    return df_out, feature_cols


# 1. 0.5초 초과 공백 기준 Segment 분리 및 5-Step Rolling Feature 생성
norm_feat, feature_cols = generate_rolling_features(
    normal_df, w_size=window_size, gap_threshold=0.5
)
out_feat, _ = generate_rolling_features(
    outlier_df, w_size=window_size, gap_threshold=0.5
)

# 2. 시간순 Hold-out Split (60% / 20% / 20%)
n_norm, n_out = len(norm_feat), len(out_feat)
tr_n_end, val_n_end = int(n_norm * 0.6), int(n_norm * 0.8)
tr_o_end, val_o_end = int(n_out * 0.6), int(n_out * 0.8)

norm_tr, norm_val, norm_te = (
    norm_feat.iloc[:tr_n_end].copy(),
    norm_feat.iloc[tr_n_end:val_n_end].copy(),
    norm_feat.iloc[val_n_end:].copy(),
)
out_tr, out_val, out_te = (
    out_feat.iloc[:tr_o_end].copy(),
    out_feat.iloc[tr_o_end:val_o_end].copy(),
    out_feat.iloc[val_o_end:].copy(),
)

# 3. 모델 입력 데이터셋 구축
X_tr_norm = norm_tr[feature_cols].values

df_tr_xgb_combined = pd.concat(
    [norm_tr[feature_cols], out_tr[feature_cols]]
).reset_index(drop=True)
y_tr_xgb = np.array([0] * len(norm_tr) + [1] * len(out_tr))
X_tr_xgb = df_tr_xgb_combined.values


X_val_combined = pd.concat(
    [norm_val[feature_cols], out_val[feature_cols]]
).reset_index(drop=True).values
y_val_combined = np.array([0] * len(norm_val) + [1] * len(out_val))

X_te_norm = norm_te[feature_cols].values
X_te_out = out_te[feature_cols].values
X_te_combined = np.vstack([X_te_norm, X_te_out])
y_te_combined = np.array([0] * len(norm_te) + [1] * len(out_te))

# 4. 정상 Train 기준 StandardScaler 표준화
scaler = StandardScaler()
X_tr_norm_scaled = scaler.fit_transform(X_tr_norm)

X_tr_xgb_scaled = scaler.transform(X_tr_xgb)
X_val_combined_scaled = scaler.transform(X_val_combined)
X_te_combined_scaled = scaler.transform(X_te_combined)

print(' TimeStamp 기반 Segment 분리 및 Rolling Feature 생성 완료')
print(f'모델 입력 피처: 센서 시계열 기반 파생변수 {len(feature_cols)}개')
print(f' - Train (Norm): {X_tr_norm_scaled.shape}')
print(f' - Train (XGB): {X_tr_xgb_scaled.shape} | 라벨 y_tr: {y_tr_xgb.shape}')
print(f' - Val (Combined): {X_val_combined_scaled.shape} | 라벨 y_val: {y_val_combined.shape}')
print(f' - Test (Combined): {X_te_combined_scaled.shape}')

 TimeStamp 기반 Segment 분리 및 Rolling Feature 생성 완료
모델 입력 피처: 센서 시계열 기반 파생변수 15개
 - Train (Norm): (10591, 15)
 - Train (XGB): (10901, 15) | 라벨 y_tr: (10901,)
 - Val (Combined): (3633, 15) | 라벨 y_val: (3633,)
 - Test (Combined): (3635, 15)


In [3]:
# [Cell 1.5] Window Size(3, 5, 10) Validation 비교 및 최종 5-Step 선정 근거 검증
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

print(
    '===  Window Size(3, 5, 10) Validation 성능 비교 실험 (XGBoost &'
    ' Isolation Forest) ==='
)
window_candidates = [3, 5, 10]
comp_results = []

for w in window_candidates:
    # 각 Window 크기 별로 0.5초 초과 공백 기준 Segment 분리 후 Rolling Feature 생성
    n_f, f_cols = generate_rolling_features(
        normal_df, w_size=w, gap_threshold=0.5
    )
    o_f, _ = generate_rolling_features(outlier_df, w_size=w, gap_threshold=0.5)

    n_n, n_o = len(n_f), len(o_f)
    n_tr_e, n_val_e = int(n_n * 0.6), int(n_n * 0.8)
    o_tr_e, o_val_e = int(n_o * 0.6), int(n_o * 0.8)

    tr_n, val_n = n_f.iloc[:n_tr_e], n_f.iloc[n_tr_e:n_val_e]
    tr_o, val_o = o_f.iloc[:o_tr_e], o_f.iloc[o_tr_e:o_val_e]

    # Train / Val 세트 구축
    X_tr_norm = tr_n[f_cols].values
    X_tr_xgb = pd.concat([tr_n[f_cols], tr_o[f_cols]]).values
    y_tr_xgb = np.array([0] * len(tr_n) + [1] * len(tr_o))

    X_val = pd.concat([val_n[f_cols], val_o[f_cols]]).values
    y_val = np.array([0] * len(val_n) + [1] * len(val_o))

    # 정상 Train 기준 StandardScaler 표준화
    sc = StandardScaler()
    X_tr_norm_sc = sc.fit_transform(X_tr_norm)
    X_tr_xgb_sc = sc.transform(X_tr_xgb)
    X_val_sc = sc.transform(X_val)

    # 1. XGBoost Validation 성능 검증
    xgb_exp = XGBClassifier(
        n_estimators=100, max_depth=4, random_state=42, eval_metric='logloss'
    )
    xgb_exp.fit(X_tr_xgb_sc, y_tr_xgb)
    xgb_preds = xgb_exp.predict(X_val_sc)

    # 2. Isolation Forest 주 탐지모델 Validation 성능 검증
    if_exp = IsolationForest(contamination=0.03, random_state=42)
    if_exp.fit(X_tr_norm_sc)
    if_preds = (if_exp.predict(X_val_sc) == -1).astype(int)

    # XGBoost Precision / Recall / F1 산출
    xgb_prec = precision_score(y_val, xgb_preds, zero_division=0)
    xgb_rec = recall_score(y_val, xgb_preds, zero_division=0)
    xgb_f1 = f1_score(y_val, xgb_preds, zero_division=0)

    # Isolation Forest Precision / Recall / F1 산출
    if_prec = precision_score(y_val, if_preds, zero_division=0)
    if_rec = recall_score(y_val, if_preds, zero_division=0)
    if_f1 = f1_score(y_val, if_preds, zero_division=0)

    comp_results.append({
        'Window Size': f'{w}-Step',
        'XGB Precision': round(xgb_prec, 4),
        'XGB Recall': round(xgb_rec, 4),
        'XGB F1': round(xgb_f1, 4),
        'IF Precision (주모델)': round(if_prec, 4),
        'IF Recall (주모델)': round(if_rec, 4),
        'IF F1 (주모델)': round(if_f1, 4),
        'Val 데이터 수': len(X_val_sc),
    })

df_window_comp = pd.DataFrame(comp_results)
print(
    '\n[Window Size별 Validation 성능 비교 '
    '(XGBoost vs Isolation Forest)]'
)
display(df_window_comp)

  # 최종 5-Step 선정:
  # XGBoost의 Validation F1이 가장 높고
  # 10-Step 대비 더 적은 관측값으로 판단할 수 있어 현장 반응성을 함께 고려함.



===  Window Size(3, 5, 10) Validation 성능 비교 실험 (XGBoost & Isolation Forest) ===

[Window Size별 Validation 성능 비교 (XGBoost vs Isolation Forest)]


,Window Size,XGB Precision,XGB Recall,XGB F1,IF Precision (주모델),IF Recall (주모델),IF F1 (주모델),Val 데이터 수
0,3-Step,0.9538,0.5536,0.7006,0.2040,0.4554,0.2818,3874
1,5-Step,1.0000,0.6699,0.8023,0.2177,0.5728,0.3155,3633
2,10-Step,1.0000,0.6512,0.7887,0.2833,0.7674,0.4138,3060


In [4]:
# [Cell 2] 다중 모델 학습, Validation Threshold 최적화 및 Test 성능평가
import tensorflow as tf
from xgboost import XGBClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

# Train / Validation 데이터 재구성 및 차원 정합성 확인
X_tr_xgb_df = pd.concat([norm_tr[feature_cols], out_tr[feature_cols]]).reset_index(drop=True)
y_tr_xgb = np.array([0] * len(norm_tr) + [1] * len(out_tr))
X_tr_xgb = X_tr_xgb_df.values

X_val_xgb_df = pd.concat([norm_val[feature_cols], out_val[feature_cols]]).reset_index(drop=True)
y_val_combined = np.array([0] * len(norm_val) + [1] * len(out_val))
X_val_combined = X_val_xgb_df.values

# 정상 Train에 적합된 StandardScaler를 동일하게 적용
X_tr_xgb_scaled = scaler.transform(X_tr_xgb)
X_val_combined_scaled = scaler.transform(X_val_combined)
X_te_combined_scaled = scaler.transform(X_te_combined)

print(f" 데이터 정합성 확인 완료 | X_tr_xgb: {X_tr_xgb_scaled.shape} | y_tr_xgb: {y_tr_xgb.shape}")

# 1. Isolation Forest 학습 (주 이상탐지 모델)
if_model = IsolationForest(contamination=0.03, random_state=42)
if_model.fit(X_tr_norm_scaled)

# 2. Autoencoder 학습 (정상 패턴 이탈도 산출)
inp_dim = X_tr_norm_scaled.shape[1]
inp = tf.keras.layers.Input(shape=(inp_dim,))
enc = tf.keras.layers.Dense(16, activation="relu")(inp)
bot = tf.keras.layers.Dense(4, activation="relu")(enc)
dec = tf.keras.layers.Dense(16, activation="relu")(bot)
out = tf.keras.layers.Dense(inp_dim, activation="linear")(dec)

ae_model = tf.keras.models.Model(inputs=inp, outputs=out)
ae_model.compile(optimizer='adam', loss='mse')
ae_model.fit(X_tr_norm_scaled, X_tr_norm_scaled, epochs=25, batch_size=64, verbose=0)

tr_ae_mse = np.mean(np.square(X_tr_norm_scaled - ae_model.predict(X_tr_norm_scaled, verbose=0)), axis=1)
ae_min, ae_max = float(tr_ae_mse.min()), float(tr_ae_mse.max())

# 3. XGBoost 학습 (지도학습 기반 보조 판별모델)
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='logloss')
xgb_model.fit(X_tr_xgb_scaled, y_tr_xgb)

# 4. Validation 데이터 기반 Hybrid Anomaly Score 임계값 결정
val_xgb_prob = xgb_model.predict_proba(X_val_combined_scaled)[:, 1]
val_ae_mse = np.mean(np.square(X_val_combined_scaled - ae_model.predict(X_val_combined_scaled, verbose=0)), axis=1)
val_ae_norm = np.clip((val_ae_mse - ae_min) / (ae_max - ae_min + 1e-8), 0, 1)

val_hybrid_score = 0.5 * val_xgb_prob + 0.5 * val_ae_norm

precisions, recalls, thresholds = precision_recall_curve(y_val_combined, val_hybrid_score)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_thresh = float(thresholds[best_idx])

# 5. 미사용 Test 데이터 최종 성능평가
te_xgb_prob = xgb_model.predict_proba(X_te_combined_scaled)[:, 1]
te_ae_mse = np.mean(np.square(X_te_combined_scaled - ae_model.predict(X_te_combined_scaled, verbose=0)), axis=1)
te_ae_norm = np.clip((te_ae_mse - ae_min) / (ae_max - ae_min + 1e-8), 0, 1)
te_hybrid_score = 0.5 * te_xgb_prob + 0.5 * te_ae_norm

pred_if = (if_model.predict(X_te_combined_scaled) == -1).astype(int)
pred_ae = (te_ae_norm > 0.5).astype(int)
pred_xgb = (te_xgb_prob > 0.5).astype(int)
pred_hybrid = (te_hybrid_score > best_thresh).astype(int)

# 모델별 역할 및 예측값 구성
models = ['Isolation Forest (주 탐지)', 'Autoencoder (정상이탈도)', 'XGBoost (보조 검증)', 'Hybrid (종합 위험도)']
preds = [pred_if, pred_ae, pred_xgb, pred_hybrid]
scores = [if_model.score_samples(X_te_combined_scaled)*-1, te_ae_norm, te_xgb_prob, te_hybrid_score]

comp_data = []
for m_name, p, s in zip(models, preds, scores):
    fp = int(np.sum((p == 1) & (y_te_combined == 0)))
    fn = int(np.sum((p == 0) & (y_te_combined == 1)))
    comp_data.append({
        'Model Pipeline': m_name,
        'Precision': round(precision_score(y_te_combined, p, zero_division=0), 4),
        'Recall': round(recall_score(y_te_combined, p, zero_division=0), 4),
        'F1-Score': round(f1_score(y_te_combined, p, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_te_combined, s), 4),
        'FP (오탐)': fp,
        'FN (미탐)': fn
    })

print(f"Validation 기반 Hybrid Anomaly Score 최적 임계값: {best_thresh:.4f}")
print("\n=== [다중 이상탐지 모델 Test 성능 비교] ===")
print(pd.DataFrame(comp_data).to_string(index=False))

# 6. XGBoost 기반 센서 Feature Importance 산출
df_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n=== [XGBoost 주요 파생변수 Feature Importance Top 10] ===")
print(df_imp.head(10).to_string(index=False))

 데이터 정합성 확인 완료 | X_tr_xgb: (10901, 15) | y_tr_xgb: (10901,)
Validation 기반 Hybrid Anomaly Score 최적 임계값: 0.3278

=== [다중 이상탐지 모델 Test 성능 비교] ===
         Model Pipeline  Precision  Recall  F1-Score  ROC-AUC  FP (오탐)  FN (미탐)
Isolation Forest (주 탐지)     0.8228  0.6250    0.7104   0.9898       14       39
    Autoencoder (정상이탈도)     0.9362  0.4231    0.5828   0.9942        3       60
        XGBoost (보조 검증)     1.0000  0.3654    0.5352   0.9524        0       66
        Hybrid (종합 위험도)     0.9348  0.4135    0.5733   0.9943        3       61

=== [XGBoost 주요 파생변수 Feature Importance Top 10] ===
                Feature  Importance
AI1_Vibration_roll_mean    0.587528
      AI0_Vibration_rms    0.299757
        AI2_Current_rms    0.026430
        AI2_Current_ptp    0.015496
 AI1_Vibration_roll_std    0.015437
      AI1_Vibration_rms    0.011323
   AI2_Current_roll_std    0.010976
 AI0_Vibration_roll_std    0.006253
AI0_Vibration_roll_mean    0.006088
  AI2_Current_roll_mean    0.005367


In [5]:
# [Cell 3] Segment 기반 Temporal Persistence 적용 및 Test 경보 안정화 성능 검증
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score


# 1. Segment 경계를 유지하는 Temporal Persistence 함수
def apply_segment_persistence(df_with_segment, raw_preds, N=5, K=3):
    df_temp = df_with_segment.copy()
    df_temp['raw_pred'] = raw_preds
    df_temp['pers_pred'] = 0

    # 각 Segment 내부에서 최근 N개 판단 중 K회 이상 이상일 때 최종 경보 발생
    for seg, group in df_temp.groupby('segment'):
        seg_preds = group['raw_pred'].values
        seg_pers = np.zeros_like(seg_preds)
        for i in range(len(seg_preds)):
            window = seg_preds[max(0, i - N + 1) : i + 1]
            if np.sum(window) >= K:
                seg_pers[i] = 1
        df_temp.loc[group.index, 'pers_pred'] = seg_pers

    return df_temp['pers_pred'].values


# Test 데이터에 Segment 정보를 결합하여 Persistence 적용 준비
df_te_combined_meta = pd.concat([norm_te, out_te]).reset_index(drop=True)

te_pred_norm_raw = pred_hybrid[: len(norm_te)]
te_pred_out_raw = pred_hybrid[len(norm_te) :]

# 정상/이상 Test 세션에 각각 Segment 기반 Persistence 적용
pred_persistence_norm = apply_segment_persistence(
    norm_te, te_pred_norm_raw, N=5, K=3
)
pred_persistence_out = apply_segment_persistence(
    out_te, te_pred_out_raw, N=5, K=3
)

pred_persistence = np.concatenate(
    [pred_persistence_norm, pred_persistence_out]
)

# 2. Persistence 적용 전후 경보 성능 비교
pers_metrics = []
for name, p in zip(
    [
        'Raw Hybrid (기본 위험도)',
        'Hybrid + Persistence (알람 안정화 옵션)',
    ],
    [pred_hybrid, pred_persistence],
):
    fp = int(np.sum((p == 1) & (y_te_combined == 0)))
    fn = int(np.sum((p == 0) & (y_te_combined == 1)))
    pers_metrics.append({
        'Pipeline Strategy': name,
        'Precision': round(precision_score(y_te_combined, p), 4),
        'Recall': round(recall_score(y_te_combined, p), 4),
        'F1-Score': round(f1_score(y_te_combined, p), 4),
        'FP (오경보)': fp,
        'FN (미탐지)': fn,
    })

print('=== [Segment별 Persistence 적용 전/후 알람 안정화 비교표] ===')
print(pd.DataFrame(pers_metrics).to_string(index=False))

# 3. 이상 Test 세션의 첫 유효 5-Step 진단 예시 확인
print(
    '\n=== [이상 Test 세션 첫 유효 5-Step 진단 예시] ==='
)

# 이상 Test 구간 시작 시점을 원본 이상 데이터의 TimeStamp와 매핑
first_test_outlier_time = out_te['TimeStamp'].iloc[0]
outlier_raw_sorted = (
    outlier_df.copy().sort_values('TimeStamp').reset_index(drop=True)
)
outlier_raw_sorted['TimeStamp'] = pd.to_datetime(
    outlier_raw_sorted['TimeStamp']
)

# 이상 Test 시작 시점부터 연속 5개 Raw 관측값 추출
test_start_idx = outlier_raw_sorted[
    outlier_raw_sorted['TimeStamp'] >= first_test_outlier_time
].index[0]
test_outlier_sample = outlier_raw_sorted.iloc[
    test_start_idx : test_start_idx + window_size
].copy()

# 동일한 Segment/5-Step 전처리를 적용하여 Feature 생성
sub_feat, _ = generate_rolling_features(
    test_outlier_sample, w_size=window_size, gap_threshold=0.5
)

if len(sub_feat) > 0:
    step_scaled = scaler.transform(sub_feat[feature_cols].values)

    step_prob = float(xgb_model.predict_proba(step_scaled)[:, 1][0])
    step_mse = float(
        np.mean(
            np.square(step_scaled - ae_model.predict(step_scaled, verbose=0))
        )
    )
    step_ae_norm = float(
        np.clip((step_mse - ae_min) / (ae_max - ae_min + 1e-8), 0.0, 1.0)
    )
    step_hybrid = float(0.5 * step_prob + 0.5 * step_ae_norm)
    if_pred = if_model.predict(step_scaled)[0]

    status_str = (
        '🔴 이상 위험'
        if (step_hybrid > best_thresh or if_pred == -1)
        else '🟢 정상 범위'
    )

    print(
        f'· 관측 상태: 연속 {window_size}-Step Feature 생성 완료 '
        f'(모델 입력 Feature: {len(feature_cols)}개)'
    )
    print(
        '• Isolation Forest 1차 판정:'
        f" {'🔴 Outlier' if if_pred == -1 else '🟢 Normal'}"
    )
    print(
        f'• Hybrid 종합 위험도 점수: {round(step_hybrid * 100, 1)} / 100점 (AE'
        f' MSE: {step_mse:.4f}, XGB Prob: {step_prob:.4f})'
    )
    print(f'• 최종 진단 결과: {status_str}')
else:
    print('⚠️ 0.5초 초과 공백으로 인해 연속 5-Step 미충족 (진단 보류)')

=== [Segment별 Persistence 적용 전/후 알람 안정화 비교표] ===
               Pipeline Strategy  Precision  Recall  F1-Score  FP (오경보)  FN (미탐지)
             Raw Hybrid (기본 위험도)     0.9348  0.4135    0.5733         3        61
Hybrid + Persistence (알람 안정화 옵션)     1.0000  0.3558    0.5248         0        67

=== [이상 Test 세션 첫 유효 5-Step 진단 예시] ===
· 관측 상태: 연속 5-Step Feature 생성 완료 (모델 입력 Feature: 15개)
• Isolation Forest 1차 판정: 🟢 Normal
• Hybrid 종합 위험도 점수: 16.9 / 100점 (AE MSE: 1.1803, XGB Prob: 0.0506)
• 최종 진단 결과: 🟢 정상 범위


In [6]:
# [Cell 3.5] Test 예측결과 생성 및 prediction_result.csv 저장
import numpy as np
import pandas as pd

# 1. Test 데이터의 TimeStamp 메타정보 결합
df_te_meta = pd.concat([norm_te, out_te]).reset_index(drop=True)

# 2. Isolation Forest Anomaly Score 계산 (높을수록 이상 패턴)
if_scores = if_model.score_samples(X_te_combined_scaled) * -1

# 3. Test 예측결과 제출용 DataFrame 구성
df_prediction_result = pd.DataFrame({
    'TimeStamp': (
        df_te_meta['TimeStamp']
        if 'TimeStamp' in df_te_meta.columns
        else df_te_meta.index
    ),
    'True_Label': y_te_combined,  # 실제 Label (0: 정상, 1: 이상)
    'IF_Pred': pred_if,  # Isolation Forest 판정 (0: 정상, 1: 이상)
    'IF_Score': np.round(if_scores, 4),  # IF 이상 점수
    'AE_MSE': np.round(te_ae_mse, 4),  # Autoencoder 복원오차
    'XGB_Prob': np.round(te_xgb_prob, 4),  # XGBoost 이상 확률
    'Hybrid_Score': np.round(te_hybrid_score, 4),  # 종합 이상 위험도 점수
    'Hybrid_Pred': pred_hybrid,  # Validation 임계값 기반 Hybrid 판정
    'Persistence_Pred': pred_persistence,  # Persistence 적용 후 최종 경보
})

# 4. CSV 파일 저장 (한글 깨짐 방지 utf-8-sig 적용)
output_filename = 'prediction_result.csv'
df_prediction_result.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(
    f"✅ 테스트 데이터 예측 결과 저장 완료: '{output_filename}'"
    f' ({len(df_prediction_result):,}행)'
)
print('\n=== [제출용 prediction_result.csv 상위 5개 행 미리보기] ===')
display(df_prediction_result.head())
print('\n=== [prediction_result.csv 하위 5개 행 미리보기] ===')
display(df_prediction_result.tail())

✅ 테스트 데이터 예측 결과 저장 완료: 'prediction_result.csv' (3,635행)

=== [제출용 prediction_result.csv 상위 5개 행 미리보기] ===


,TimeStamp,True_Label,IF_Pred,IF_Score,AE_MSE,XGB_Prob,Hybrid_Score,Hybrid_Pred,Persistence_Pred
0,2022-07-12 01:01:00.932,0,0,0.4196,0.0951,0.0,0.0108,0,0
1,2022-07-12 01:01:01.032,0,0,0.4091,0.0716,0.0,0.0079,0,0
2,2022-07-12 01:01:01.132,0,0,0.4103,0.0361,0.0,0.0036,0,0
3,2022-07-12 01:01:01.232,0,0,0.4574,0.0330,0.0,0.0032,0,0
4,2022-07-12 01:01:01.332,0,0,0.4337,0.1410,0.0,0.0164,0,0



=== [prediction_result.csv 하위 5개 행 미리보기] ===


,TimeStamp,True_Label,IF_Pred,IF_Score,AE_MSE,XGB_Prob,Hybrid_Score,Hybrid_Pred,Persistence_Pred
3630,2022-07-17 10:53:53.140,1,0,0.5287,0.3723,0.0002,0.0449,0,0
3631,2022-07-17 10:53:53.240,1,1,0.5637,0.5074,0.0005,0.0617,0,0
3632,2022-07-17 10:53:53.340,1,1,0.5590,0.5117,0.0007,0.0623,0,0
3633,2022-07-17 10:53:53.440,1,1,0.5717,0.5306,0.0008,0.0647,0,0
3634,2022-07-17 10:53:53.540,1,1,0.5782,0.5095,0.0004,0.0619,0,0


In [7]:
# [Cell 4] 주 탐지모델(Isolation Forest) 기준 FP/FN 특징량 비교 분석
df_te_raw = pd.concat([norm_te, out_te]).reset_index(drop=True)
df_te_raw['y_true'] = y_te_combined
df_te_raw['y_pred'] = pred_if  # Isolation Forest 기준 분석

df_tp = df_te_raw[(df_te_raw['y_true'] == 1) & (df_te_raw['y_pred'] == 1)]
df_fn = df_te_raw[(df_te_raw['y_true'] == 1) & (df_te_raw['y_pred'] == 0)]
df_fp = df_te_raw[(df_te_raw['y_true'] == 0) & (df_te_raw['y_pred'] == 1)]
df_tn = df_te_raw[(df_te_raw['y_true'] == 0) & (df_te_raw['y_pred'] == 0)]

print("=== [Isolation Forest Test 오류유형 분류] ===")
print(f"TP(탐지 성공): {len(df_tp)}건 | TN(정상 판정): {len(df_tn)}건")
print(f"FP(오탐 발생): {len(df_fp)}건 | FN(미탐 발생): {len(df_fn)}건")

print("\n=== [TP(탐지 성공) vs FN(미탐) 센서 파생변수 평균 비교] ===")
key_features = ['AI0_Vibration_rms', 'AI0_Vibration_ptp', 'AI1_Vibration_rms', 'AI1_Vibration_ptp', 'AI2_Current_rms', 'AI2_Current_roll_std']

if len(df_fn) > 0:
    tp_means = df_tp[key_features].mean()
    fn_means = df_fn[key_features].mean()

    df_fn_compare = pd.DataFrame({
        'Key Feature': key_features,
        'TP Mean (탐지 성공)': [round(v, 4) for v in tp_means],
        'FN Mean (미탐지)': [round(v, 4) for v in fn_means]
    })
    print(df_fn_compare.to_string(index=False))
    print("\n[FN 미탐지 특성 해석]")
    print("-> FN 샘플은 TP 샘플 대비 일부 진동 크기 및 전류 변동 관련 파생변수가 낮게 나타남.")
    print("   이는 상대적으로 변화폭이 작은 이상구간에서 모델의 탐지 민감도가 낮아질 수 있음을 시사함.")

print("\n=== [TN(정상) vs FP(오탐) 센서 파생변수 평균 비교] ===")
if len(df_fp) > 0:
    tn_means = df_tn[key_features].mean()
    fp_means = df_fp[key_features].mean()

    df_fp_compare = pd.DataFrame({
        'Key Feature': key_features,
        'TN Mean (정상 판정)': [round(v, 4) for v in tn_means],
        'FP Mean (오탐 발생)': [round(v, 4) for v in fp_means]
    })
    print(df_fp_compare.to_string(index=False))
    print("\n[FP 오경보 특성 해석]")
    print("-> FP 샘플은 TN 샘플 대비 진동 및 전류 관련 파생변수가 상대적으로 크게 나타남.")
    print("   이는 정상상태의 일시적 센서 변동이 이상패턴과 유사하게 나타날 경우 오경보가 발생할 수 있음을 시사함.")

=== [Isolation Forest Test 오류유형 분류] ===
TP(탐지 성공): 65건 | TN(정상 판정): 3517건
FP(오탐 발생): 14건 | FN(미탐 발생): 39건

=== [TP(탐지 성공) vs FN(미탐) 센서 파생변수 평균 비교] ===
         Key Feature  TP Mean (탐지 성공)  FN Mean (미탐지)
   AI0_Vibration_rms           0.3442         0.0527
   AI0_Vibration_ptp           0.7247         0.1075
   AI1_Vibration_rms           0.1947         0.0719
   AI1_Vibration_ptp           0.3453         0.1180
     AI2_Current_rms         275.0539       183.3187
AI2_Current_roll_std          58.4314        55.8285

[FN 미탐지 특성 해석]
-> FN 샘플은 TP 샘플 대비 일부 진동 크기 및 전류 변동 관련 파생변수가 낮게 나타남.
   이는 상대적으로 변화폭이 작은 이상구간에서 모델의 탐지 민감도가 낮아질 수 있음을 시사함.

=== [TN(정상) vs FP(오탐) 센서 파생변수 평균 비교] ===
         Key Feature  TN Mean (정상 판정)  FP Mean (오탐 발생)
   AI0_Vibration_rms           0.0479           0.1144
   AI0_Vibration_ptp           0.1244           0.3041
   AI1_Vibration_rms           0.0603           0.1902
   AI1_Vibration_ptp           0.1587           0.5225
     AI2_Current_rms          94.5812 

In [8]:
# [Cell 5] Gradio 기반 현장 진단 및 연속 모니터링 UI
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gradio as gr

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False


# 1. Mode 1: 실시간 5-Step 현장 진단
def diagnose_single_sequence(time_series_df):
    try:
        if isinstance(time_series_df, pd.DataFrame):
            ts_data = time_series_df.copy()
        else:
            ts_data = pd.DataFrame(time_series_df)

        for col in sensor_cols:
            if col in ts_data.columns:
                ts_data[col] = (
                    pd.to_numeric(ts_data[col], errors='coerce')
                    .fillna(0.0)
                    .astype(float)
                )
            else:
                ts_data[col] = 0.0

        if len(ts_data) < window_size:
            return (
                '🟡 진단 보류 (데이터 수집 중)',
                0.0,
                '0점',
                None,
                {'수집 진행 중': 1.0},
                f'• 연속된 {window_size}개 관측값 수집 필요 (현재 {len(ts_data)}개 수집됨)',
            )

        ts_feat, _ = generate_rolling_features(
            ts_data, w_size=window_size, gap_threshold=0.5
        )

        if len(ts_feat) == 0:
            return (
                '🟡 진단 보류 (시간 공백 감지)',
                0.0,
                '0점',
                None,
                {'공백 발생': 1.0},
                '• 수집 데이터 내 0.5초 초과 공백이 감지되어 Rolling 버퍼가 초기화되었습니다.\n• 재개 후 연속 5-Step 확보 시 진단이 자동 개시됩니다.',
            )

        latest_sample = ts_feat[feature_cols].iloc[[-1]]
        sample_scaled = scaler.transform(latest_sample.values)

        if_pred_val = if_model.predict(sample_scaled)[0]

        ae_rec = ae_model.predict(sample_scaled, verbose=0)
        raw_mse = float(np.mean(np.square(sample_scaled - ae_rec)))
        raw_prob = float(xgb_model.predict_proba(sample_scaled)[:, 1][0])

        ae_norm = float(
            np.clip((raw_mse - ae_min) / (ae_max - ae_min + 1e-8), 0.0, 1.0)
        )
        hybrid_score = float(0.5 * raw_prob + 0.5 * ae_norm)

        # Hybrid Anomaly Score와 Isolation Forest 결과를 이용한 2단계 상태 판정
        status_str = (
            '🔴 이상 위험'
            if (hybrid_score > float(best_thresh) or if_pred_val == -1)
            else '🟢 정상 범위'
        )

        # Autoencoder 재구성오차를 센서별로 합산하여 상대적 이상 기여도 산출
        # 고장원인 확정이 아닌 점검 우선순위용 보조정보로 활용
        diff = np.abs(sample_scaled[0] - ae_rec[0])
        ai0_c = float(
            np.sum(diff[[i for i, c in enumerate(feature_cols) if 'AI0' in c]])
        )
        ai1_c = float(
            np.sum(diff[[i for i, c in enumerate(feature_cols) if 'AI1' in c]])
        )
        ai2_c = float(
            np.sum(diff[[i for i, c in enumerate(feature_cols) if 'AI2' in c]])
        )
        tot = ai0_c + ai1_c + ai2_c + 1e-8

        c_ratio = {
            'AI0_Vibration': float(round(ai0_c / tot, 2)),
            'AI1_Vibration': float(round(ai1_c / tot, 2)),
            'AI2_Current': float(round(ai2_c / tot, 2)),
        }

        fig, ax = plt.subplots(figsize=(6, 2.5), dpi=100)
        steps = [f'Step {i+1}' for i in range(len(ts_data))]
        ax.plot(
            steps,
            ts_data['AI0_Vibration'],
            marker='o',
            label='AI0 Vibration',
            color='#1f77b4',
            linewidth=2,
        )
        ax.plot(
            steps,
            ts_data['AI1_Vibration'],
            marker='s',
            label='AI1 Vibration',
            color='#ff7f0e',
            linewidth=2,
        )
        ax2 = ax.twinx()
        ax2.plot(
            steps,
            ts_data['AI2_Current'],
            marker='^',
            label='AI2 Current',
            color='#2ca02c',
            linestyle='--',
            linewidth=1.5,
        )
        ax.set_title(
            f'Recent {window_size}-Step Sensor Trends',
            fontsize=10,
            fontweight='bold',
        )
        ax.set_ylabel('Vibration (g)', fontsize=8)
        ax2.set_ylabel('Current (A)', fontsize=8)
        ax.grid(True, linestyle=':', alpha=0.6)

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(
            lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=7
        )
        plt.tight_layout()

        risk_score_str = f'{round(hybrid_score * 100, 1)} / 100점'

        max_sensor = max(c_ratio, key=c_ratio.get)
        if hybrid_score > float(best_thresh) or if_pred_val == -1:
            if 'AI1' in max_sensor:
                action_detail = (
                    '• [AI1_Vibration 이상 기여도 상위]\n  - 해당 진동 센서 및 관련 펌프/모터 계통의 체결·진동 상태 우선 확인'
                )
            elif 'AI0' in max_sensor:
                action_detail = (
                    '• [AI0_Vibration 이상 기여도 상위]\n  - 해당 진동 센서 및 관련 펌프/모터 계통의 체결·진동 상태 우선 확인'
                )
            else:
                action_detail = (
                    '• [AI2_Current 이상 기여도 상위]\n  - 해당 전류 센서 및 모터 구동 계통의 전류·운전 상태 우선 확인'
                )
            rec_actions = (
                f'{action_detail}\n• 이상 상태 지속 시 설비 담당자의 현장 정밀 점검 실시'
            )
        else:
            rec_actions = (
                '• 모든 센서 수치 및 패턴 정상 범위 유지 중\n• 특이사항 없음. 정상 가동 유지'
            )

        return (
            status_str,
            float(round(raw_mse, 4)),
            str(risk_score_str),
            fig,
            c_ratio,
            str(rec_actions),
        )

    except Exception as e:
        return (
            f'진단 오류: {str(e)}',
            0.0,
            '0점',
            None,
            {'오류': 1.0},
            f'실시간 진단 처리 중 예외 발생: {str(e)}',
        )


# 2. Mode 2: 정상·이상 세션 기반 연속 모니터링 검증
def verify_continuous_monitoring(dataset_choice, file_obj=None):
    if dataset_choice == '정상 운전 세션 검증 (press_data_normal.csv)':
        df_raw = normal_df.copy()
        is_normal_session = True
    elif dataset_choice == '이상 상태 세션 검증 (outlier_data.csv)':
        df_raw = outlier_df.copy()
        is_normal_session = False
    else:
        if file_obj is None:
            return 'CSV 파일을 업로드해 주세요.', None, '분석 데이터 없음', ''
        df_raw = pd.read_csv(file_obj.name)
        is_normal_session = True

    for col in sensor_cols:
        if col not in df_raw.columns:
            return (
                f'필수 센서 칼럼({col})이 존재하지 않습니다.',
                None,
                '오류 발생',
                '',
            )

    df_feat, _ = generate_rolling_features(
        df_raw, w_size=window_size, gap_threshold=0.5
    )
    scaled_feats = scaler.transform(df_feat[feature_cols].values)

    # 모델 연산
    if_preds = (if_model.predict(scaled_feats) == -1).astype(int)
    ae_mses = np.mean(
        np.square(scaled_feats - ae_model.predict(scaled_feats, verbose=0)),
        axis=1,
    )
    ae_norms = np.clip((ae_mses - ae_min) / (ae_max - ae_min + 1e-8), 0, 1)
    xgb_probs = xgb_model.predict_proba(scaled_feats)[:, 1]
    hybrid_scores = 0.5 * xgb_probs + 0.5 * ae_norms
    raw_alerts = (hybrid_scores > best_thresh).astype(int)

    # Segment 단위 Persistence 연산
    persistent_alerts = apply_segment_persistence(
        df_feat, raw_alerts, N=5, K=3
    )

    total_raw_steps = len(df_raw)
    total_valid_steps = len(df_feat)

    start_time = (
        str(df_raw['TimeStamp'].iloc[0])
        if 'TimeStamp' in df_raw.columns
        else 'Row 1'
    )
    end_time = (
        str(df_raw['TimeStamp'].iloc[-1])
        if 'TimeStamp' in df_raw.columns
        else f'Row {total_raw_steps}'
    )

    first_if_idx = np.where(if_preds == 1)[0]
    first_pers_idx = np.where(persistent_alerts == 1)[0]

    if_row_str = (
        f'유효 Row {first_if_idx[0] + 1}' if len(first_if_idx) > 0 else '미발생'
    )
    pers_row_str = (
        f'유효 Row {first_pers_idx[0] + 1} 확정'
        if len(first_pers_idx) > 0
        else '미발생'
    )
    pers_ratio = round(np.mean(persistent_alerts) * 100, 2)

    # 세션별 모니터링 결과 요약
    if is_normal_session:
        summary_text = (
            '📌 정상 운전 세션 (2022-07-12) — 오경보 억제 검증\n'
            '--------------------------------------------------\n'
            f'• 측정 구간: {start_time} ~ {end_time}\n'
            f'• 원본 데이터 수집량: {total_raw_steps:,} 개 행\n'
            f'• 5-Step 유효 분석 타임스텝: {total_valid_steps:,} 개 시점\n'
            f'• IF 단발성 이상징후 발생: 있음 (일시적 이상징후 확인)\n'
            f'• Persistence 최종 알람: {pers_row_str} (최종 오경보 미발생)\n'
            f'• 최종 오경보 알람 비율: {pers_ratio}%'
        )
    else:
        summary_text = (
            '📌 이상 상태 세션 (2022-07-17) — 이상상태 탐지 및 지속 감시 검증\n'
            '--------------------------------------------------\n'
            f'• 측정 구간: {start_time} ~ {end_time}\n'
            f'• 원본 데이터 수집량: {total_raw_steps:,} 개 행\n'
            f'• 5-Step 유효 분석 타임스텝: {total_valid_steps:,} 개 시점\n'
            f'• IF 이상징후 최초 감지: {if_row_str}\n'
            f'• Persistence 최종 알람 확정: {pers_row_str}\n'
            f'• 세션 내 알람 유지 비율: {pers_ratio}%'
        )

    # 시계열 위험도 차트
    fig, ax = plt.subplots(figsize=(9, 3.5), dpi=100)
    rows = np.arange(1, total_valid_steps + 1)

    ax.plot(
        rows,
        hybrid_scores * 100,
        label='Hybrid Anomaly Score',
        color='#1f77b4',
        linewidth=1.2,
        alpha=0.85,
    )
    ax.axhline(
        y=best_thresh * 100,
        color='red',
        linestyle='--',
        linewidth=1.2,
        label=f'Alarm Threshold ({round(best_thresh*100, 1)}점)',
    )

    if len(first_pers_idx) > 0:
        ax.axvline(
            x=first_pers_idx[0] + 1,
            color='darkred',
            linestyle=':',
            linewidth=1.5,
            label='Confirmed Alert Zone',
        )

    ax.fill_between(
        rows,
        0,
        100,
        where=(persistent_alerts == 1),
        color='red',
        alpha=0.15,
        label='🚨 Confirmed Alarm Zone',
    )

    title_str = (
        'Normal Session (False Alarm Monitoring)'
        if is_normal_session
        else 'Abnormal Session (Continuous Anomaly Monitoring)'
    )
    ax.set_title(
        f'Continuous Sensor Monitoring - {title_str}',
        fontsize=11,
        fontweight='bold',
    )
    ax.set_xlabel('Valid Time Step Index (Segment Restored)', fontsize=9)
    ax.set_ylabel('Risk Score (0~100)', fontsize=9)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(loc='upper left', fontsize=8)
    plt.tight_layout()

    res_df = pd.DataFrame({
        'Valid_Row': rows,
        'TimeStamp': (
            df_feat['TimeStamp'] if 'TimeStamp' in df_feat.columns else rows
        ),
        'AI0_Vib': df_feat['AI0_Vibration'],
        'AI1_Vib': df_feat['AI1_Vibration'],
        'AI2_Curr': df_feat['AI2_Current'],
        'IF Status': ['🔴 Outlier' if p == 1 else '🟢 Normal' for p in if_preds],
        'Risk Score': [f'{round(s*100, 1)}점' for s in hybrid_scores],
        'Persist Alert': [
            '🚨 ALARM' if a == 1 else 'NORMAL' for a in persistent_alerts
        ],
    })

    # 동적 자동 산출 비교 표
    comparison_table_md = f"""
### 📊 세션별 독립 모니터링 검증 동적 동기화 비교표
| 검증 세션 구분 | 원본 데이터 규모 | 유효 분석 데이터 (5-Step) | Persistence 최종 알람 | 알람 유지 비율 | 모델 주요 검증 목적 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **현재 검증 세션** | **{total_raw_steps:,} 행** | **{total_valid_steps:,} 행** | **{pers_row_str}** | **{pers_ratio}%** | **{'오경보 억제 검증' if is_normal_session else '이상 상태 탐지 및 지속 감시 검증'}** |
"""

    return summary_text, fig, res_df.tail(20), comparison_table_md


# 3. Mode 3: 센서 스트림 유입 상황을 가정한 현장 모니터링 시뮬레이션
def run_realtime_demo_step(target_session, step_index):
    df_src = normal_df if '정상' in target_session else outlier_df
    idx = min(int(step_index) - 1, len(df_src) - window_size)

    sub_df = df_src.iloc[idx : idx + window_size].copy()
    ts_val = (
        str(sub_df['TimeStamp'].iloc[-1])
        if 'TimeStamp' in sub_df.columns
        else f'Step {step_index}'
    )

    sub_feat, _ = generate_rolling_features(
        sub_df, w_size=window_size, gap_threshold=0.5
    )
    if len(sub_feat) == 0:
        return (
            '⚠️ 선택 구간 내 0.5초 초과 공백으로 데이터가 미충족되었습니다.',
            '🟡 데이터 수집 진행 중',
        )

    sample_scaled = scaler.transform(sub_feat[feature_cols].iloc[[-1]].values)

    if_pred = if_model.predict(sample_scaled)[0]
    if_str = '🔴 Outlier' if if_pred == -1 else '🟢 Normal'

    raw_mse = float(
        np.mean(
            np.square(sample_scaled - ae_model.predict(sample_scaled, verbose=0))
        )
    )
    raw_prob = float(xgb_model.predict_proba(sample_scaled)[:, 1][0])
    ae_norm = float(
        np.clip((raw_mse - ae_min) / (ae_max - ae_min + 1e-8), 0.0, 1.0)
    )
    hybrid_score = float(0.5 * raw_prob + 0.5 * ae_norm)

    window_start = max(0, idx - 4)
    hist_sub = df_src.iloc[window_start : idx + window_size]
    hist_feat, _ = generate_rolling_features(
        hist_sub, w_size=window_size, gap_threshold=0.5
    )

    if len(hist_feat) > 0:
        hist_scaled = scaler.transform(hist_feat[feature_cols].values)
        hist_probs = xgb_model.predict_proba(hist_scaled)[:, 1]
        hist_mses = np.mean(
            np.square(
                hist_scaled - ae_model.predict(hist_scaled, verbose=0)
            ),
            axis=1,
        )
        hist_norms = np.clip(
            (hist_mses - ae_min) / (ae_max - ae_min + 1e-8), 0, 1
        )
        hist_hybrids = 0.5 * hist_probs + 0.5 * hist_norms

        recent_alerts = (hist_hybrids[-5:] > best_thresh).astype(int)
        persist_count = int(np.sum(recent_alerts))
    else:
        persist_count = 0

    persist_status = (
        f'🚨 ALARM (지속 횟수 {persist_count}/5)'
        if persist_count >= 3
        else f'🟢 NORMAL (지속 횟수 {persist_count}/5)'
    )

    telemetry = (
        f'⏱️ Current TimeStamp: {ts_val}\n'
        '----------------------------------------\n'
        f"• AI0_Vibration : {sub_df['AI0_Vibration'].iloc[-1]:.4f} g\n"
        f"• AI1_Vibration : {sub_df['AI1_Vibration'].iloc[-1]:.4f} g\n"
        f"• AI2_Current   : {sub_df['AI2_Current'].iloc[-1]:.2f} A\n"
        '----------------------------------------\n'
        f'• 1차 주 탐지 (IF) : {if_str}\n'
        f'• 종합 위험도 점수 : {round(hybrid_score*100, 1)} / 100점\n'
        f'• Persistence 알람 : {persist_status}'
    )

    guide_text = (
        '🚨 이상 지속 발생! 현장 설비 담당자 구동계 정밀 점검 권고'
        if persist_count >= 3
        else '🟢 설비 가동 상태 정상 유지 중'
    )

    return telemetry, guide_text


default_sequence = normal_df[sensor_cols].iloc[100:105].reset_index(drop=True)

# Gradio 통합 UI
with gr.Blocks(title='Hydraulic Pump AI Monitor') as demo:
    gr.Markdown(
        '# 🛠️ 프레스 유압펌프 시계열 AI 진단 & 현장 의사결정 지원 시스템'
    )

    with gr.Tab('Mode 1: 실시간 5-Step 현장 진단'):
        gr.Markdown('### 📥 최근 5개 연속 시점 센서 입력 기반 즉시 진단')
        with gr.Row():
            with gr.Column(scale=1):
                df_in = gr.Dataframe(
                    value=default_sequence,
                    headers=['AI0_Vibration', 'AI1_Vibration', 'AI2_Current'],
                    column_count=(3, 'fixed'),
                    row_count=(5, 'fixed'),
                    label='최근 5-Step 시계열 데이터',
                )
                btn = gr.Button('🔍 실시간 시계열 진단 실행', variant='primary')

            with gr.Column(scale=1):
                status_out = gr.Textbox(
                    label='1차 주 탐지 판정 (Isolation Forest)'
                )
                with gr.Row():
                    score_out = gr.Number(label='정상 패턴 이탈 점수 (AE MSE)')
                    risk_out = gr.Textbox(
                        label='종합 이상 위험 점수 (Hybrid Anomaly Score)'
                    )
                plot_out = gr.Plot(label='최근 5-Step 센서 변동 추세')

        with gr.Row():
            with gr.Column(scale=1):
                contrib_out = gr.Label(label='주요 센서별 이상 기여도')
            with gr.Column(scale=1):
                action_out = gr.Textbox(
                    label='📋 현장 맞춤형 권장 점검 수칙 (Prescriptive Guide)',
                    lines=5,
                )

        btn.click(
            fn=diagnose_single_sequence,
            inputs=[df_in],
            outputs=[
                status_out,
                score_out,
                risk_out,
                plot_out,
                contrib_out,
                action_out,
            ],
        )

    with gr.Tab('Mode 2: 센서 시계열 연속 모니터링 검증'):
        gr.Markdown(
            '### 📂 세션별 독립 데이터 기반 연속 스캔 및 오경보/지속 탐지 검증'
        )
        with gr.Row():
            with gr.Column(scale=1):
                verify_selector = gr.Radio(
                    choices=[
                        '정상 운전 세션 검증 (press_data_normal.csv)',
                        '이상 상태 세션 검증 (outlier_data.csv)',
                        '개별 CSV 파일 업로드',
                    ],
                    value='정상 운전 세션 검증 (press_data_normal.csv)',
                    label='검증 데이터 세션 선택',
                )
                file_verify_in = gr.File(
                    label='개별 CSV 파일 업로드 (개별 선택 시)',
                    file_types=['.csv'],
                )
                verify_btn = gr.Button(
                    '▶️ 연속 모니터링 검증 실행', variant='primary'
                )
                verify_summary = gr.Textbox(
                    label='📊 모니터링 검증 결과 요약', lines=7
                )

            with gr.Column(scale=1):
                verify_plot = gr.Plot(
                    label=(
                        '세션별 이상 위험도 및 알람 확정 구간(Shaded Zone)'
                        ' 추이'
                    )
                )

        verify_logs = gr.Dataframe(
            label='최근 20개 타임스텝 세부 진단 로그'
        )
        comparison_md_out = gr.Markdown()

        verify_btn.click(
            fn=verify_continuous_monitoring,
            inputs=[verify_selector, file_verify_in],
            outputs=[
                verify_summary,
                verify_plot,
                verify_logs,
                comparison_md_out,
            ],
        )

    with gr.Tab('Mode 3: 🏭 현장 적용 실시간 모니터링 데모'):
        gr.Markdown(
            '### ⏱️ 현장 실시간 센서 스트림 유입 상황 UI 시뮬레이션'
        )
        with gr.Row():
            with gr.Column(scale=1):
                demo_session_sel = gr.Radio(
                    choices=[
                        '정상 가동 설비 모니터링 데모',
                        '이상 징후 설비 모니터링 데모',
                    ],
                    value='이상 징후 설비 모니터링 데모',
                    label='모니터링 설비 대상 선택',
                )
                # 5-Step 진단 구간의 시작 인덱스 선택
                demo_step_slider = gr.Slider(
                    minimum=1,
                    maximum=500,
                    value=1,
                    step=1,
                    label='5-Step 진단 구간 시작 인덱스',
                )
                demo_run_btn = gr.Button(
                    '🔄 해당 시점 실시간 감시 실행', variant='primary'
                )

            with gr.Column(scale=1):
                demo_telemetry_out = gr.Textbox(
                    label='📡 실시간 센서 텔레메트리 & AI 진단 현황', lines=8
                )
                demo_action_out = gr.Textbox(
                    label='📢 현장 제어/점검 알람 상태', lines=3
                )

        demo_run_btn.click(
            fn=run_realtime_demo_step,
            inputs=[demo_session_sel, demo_step_slider],
            outputs=[demo_telemetry_out, demo_action_out],
        )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://257e27d2949b5f5b4b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
